In [102]:
import yfinance as yf
import numpy as np
stock = yf.Ticker('TCS.NS')

In [103]:
data_stock = stock.history(period='1y')

In [104]:
print(data_stock.head())
print(data_stock.shape)
print(data_stock.isnull().sum())
print(data_stock.columns.to_list())


                                  Open         High          Low        Close  \
Date                                                                            
2025-05-05 00:00:00+05:30  3334.753505  3387.432945  3331.273668  3338.909912   
2025-05-06 00:00:00+05:30  3344.419922  3356.985661  3322.188229  3344.419922   
2025-05-07 00:00:00+05:30  3306.335993  3343.936456  3306.335993  3330.017578   
2025-05-08 00:00:00+05:30  3332.820588  3360.851852  3314.165252  3333.690430   
2025-05-09 00:00:00+05:30  3285.457255  3331.080503  3284.587413  3325.377686   

                            Volume  Dividends  Stock Splits  
Date                                                         
2025-05-05 00:00:00+05:30  1284443        0.0           0.0  
2025-05-06 00:00:00+05:30  1324763        0.0           0.0  
2025-05-07 00:00:00+05:30  1477128        0.0           0.0  
2025-05-08 00:00:00+05:30  2543591        0.0           0.0  
2025-05-09 00:00:00+05:30  2531757        0.0           0.0 

In [105]:
data_stock.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-05-05 00:00:00+05:30,3334.753505,3387.432945,3331.273668,3338.909912,1284443,0.0,0.0
2025-05-06 00:00:00+05:30,3344.419922,3356.985661,3322.188229,3344.419922,1324763,0.0,0.0
2025-05-07 00:00:00+05:30,3306.335993,3343.936456,3306.335993,3330.017578,1477128,0.0,0.0
2025-05-08 00:00:00+05:30,3332.820588,3360.851852,3314.165252,3333.690430,2543591,0.0,0.0
2025-05-09 00:00:00+05:30,3285.457255,3331.080503,3284.587413,3325.377686,2531757,0.0,0.0


In [106]:
import sys
sys.path.append('..')

from src.utils import get_logger

logger = get_logger("test")
print("logger created:", logger)
logger.info("hello this is a log message")

2026-05-06 00:28:11,072 - test - INFO - hello this is a log message


logger created: <Logger test (INFO)>


In [107]:
len(data_stock)

250

In [108]:
import sys
sys.path.append('..')
from src.data_ingestion import fetch_stock_data

df=fetch_stock_data('RELIANCE.NS','1y')
df

2026-05-06 00:28:11,095 - src.data_ingestion - INFO - Fetching data for RELIANCE.NS over the period of 1y
2026-05-06 00:28:11,341 - src.data_ingestion - INFO - Successfully fetched 250 rows for RELIANCE.NS


,Open,Close,Low,High
Date,,,,
2025-05-05 00:00:00+05:30,1425.307468,1425.606323,1421.223802,1433.773655
2025-05-06 00:00:00+05:30,1425.307479,1415.247681,1404.988605,1426.303501
2025-05-07 00:00:00+05:30,1415.247612,1400.406860,1397.119939,1418.733689
2025-05-08 00:00:00+05:30,1398.514466,1401.402954,1392.438756,1415.148106
2025-05-09 00:00:00+05:30,1379.988467,1371.721436,1369.032225,1389.251520
...,...,...,...,...
2026-04-29 00:00:00+05:30,1392.000000,1425.400024,1391.300049,1433.800049
2026-04-30 00:00:00+05:30,1409.000000,1430.800049,1393.099976,1437.000000
2026-05-01 00:00:00+05:30,1430.800049,1430.800049,1430.800049,1430.800049


FEATURE ENGINEERING

1 - Daily Returns

In [109]:
# This calculates the percentage change from the previous day's close
df['Daily_Return'] = df['Close'].pct_change()
df.iloc[0, df.columns.get_loc('Daily_Return')] = 0
print(f'Average daily return: {df["Daily_Return"].mean():.2%}')
print(f'Standard deviation of daily return: {df["Daily_Return"].std():.2%}')
df

Average daily return: 0.02%
Standard deviation of daily return: 1.27%


,Open,Close,Low,High,Daily_Return
Date,,,,,
2025-05-05 00:00:00+05:30,1425.307468,1425.606323,1421.223802,1433.773655,0.000000
2025-05-06 00:00:00+05:30,1425.307479,1415.247681,1404.988605,1426.303501,-0.007266
2025-05-07 00:00:00+05:30,1415.247612,1400.406860,1397.119939,1418.733689,-0.010486
2025-05-08 00:00:00+05:30,1398.514466,1401.402954,1392.438756,1415.148106,0.000711
2025-05-09 00:00:00+05:30,1379.988467,1371.721436,1369.032225,1389.251520,-0.021180
...,...,...,...,...,...
2026-04-29 00:00:00+05:30,1392.000000,1425.400024,1391.300049,1433.800049,0.026280
2026-04-30 00:00:00+05:30,1409.000000,1430.800049,1393.099976,1437.000000,0.003788
2026-05-01 00:00:00+05:30,1430.800049,1430.800049,1430.800049,1430.800049,0.000000


2 - Volatility

In [110]:
# std() gets the daily volatility. We multiply by sqrt(252) to annualise it (252 trading days in a year)
volatility = df['Daily_Return'].std() * np.sqrt(252)
print(volatility)

0.20096334821347983


3 - Sharpe Ratio

In [111]:
mean_return = df['Daily_Return'].mean()
sharpe_ratio = (mean_return * 252)/volatility
print(sharpe_ratio)

0.23085418614823566


4 - Max Drawdown

In [112]:
# First, we calculate the cumulative return over time (if you invested $1, what is it worth now?)
cumulative_return = (1+df['Daily_Return']).cumprod()

# Then we find the biggest drop from an all-time high
max_drawdown = (cumulative_return / cumulative_return.cummax() - 1).min()
print(cumulative_return)
print('Max Drawdown: ', max_drawdown)

Date
2025-05-05 00:00:00+05:30    1.000000
2025-05-06 00:00:00+05:30    0.992734
2025-05-07 00:00:00+05:30    0.982324
2025-05-08 00:00:00+05:30    0.983022
2025-05-09 00:00:00+05:30    0.962202
                               ...   
2026-04-29 00:00:00+05:30    0.999855
2026-04-30 00:00:00+05:30    1.003643
2026-05-01 00:00:00+05:30    1.003643
2026-05-04 00:00:00+05:30    1.026300
2026-05-05 00:00:00+05:30         NaN
Name: Daily_Return, Length: 250, dtype: float64
Max Drawdown:  -0.1806820727374372


SUMMARY

In [113]:
print(f"Annualised Volatility: {volatility:.2%}")
print(f"Sharpe Ratio:          {sharpe_ratio:.2f}")
print(f"Max Drawdown:          {max_drawdown:.2%}")

Annualised Volatility: 20.10%
Sharpe Ratio:          0.23
Max Drawdown:          -18.07%


In [ ]:
from src.feature_engineering import calculate_daily_returns, calculate_volatility, calculate_sharpe_ratio, calculate_max_drawdown
df_infy = fetch_stock_data('INFY.NS','1y')
df_infy

In [ ]:
daily_return = calculate_daily_returns(df_infy)
daily_return

In [123]:
volatility_val = calculate_volatility(daily_return)
volatility_val

2026-05-06 00:33:38,772 - src.feature_engineering - INFO - Calculating annualised volatility


np.float64(0.26167401725051437)

In [127]:
sharpe_ratio_val = calculate_sharpe_ratio(daily_return, volatility_val)
sharpe_ratio_val

np.float64(-0.7446497248902791)

In [128]:
max_drawdown_val = calculate_max_drawdown(daily_return)
max_drawdown_val

2026-05-06 00:36:13,114 - src.feature_engineering - INFO - Calculating Max Drawdowns


np.float64(-0.31820337182205793)

In [129]:
print("-" * 30)
print("Infosys (INFY.NS) Risk Metrics:")
print(f"Annualised Volatility: {volatility_val:.2%}")
print(f"Sharpe Ratio:          {sharpe_ratio_val:.2f}")
print(f"Max Drawdown:          {max_drawdown_val:.2%}")

------------------------------
Infosys (INFY.NS) Risk Metrics:
Annualised Volatility: 26.17%
Sharpe Ratio:          -0.74
Max Drawdown:          -31.82%
